<a href="https://colab.research.google.com/github/EthanDev/Arduino/blob/esp8266/huey_free_linkedin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install crewai==0.117.1 crewai_tools==0.43.0 langchain_community==0.3.23
!pip install composio-crewai

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
from crewai import Agent, Crew, Task
from composio_crewai import ComposioToolSet, App, Action


In [4]:
!pip install utils

  Preparing metadata (setup.py) ... done
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=13906 sha256=916c34aa587d45ba44a5c0b0e99bce7672833ff0906512e42802d9f8f62b4757
  Stored in directory: /root/.cache/pip/wheels/15/0c/b3/674aea8c5d91c642c817d4d630bd58faa316724b136844094d
Successfully built utils


In [5]:
!pip install openai

In [6]:
import os
import openai
import asyncio
import getpass
import json
import requests
import aiohttp
import time
import json
import nest_asyncio
from openai import OpenAI
from google.colab import userdata


os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') or 'OPENAI_API_KEY'
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY') or 'SERPER_API_KEY'
os.environ["COMPOSIO_API_KEY"] = userdata.get('COMPOSIO_API_KEY') or 'COMPOSIO_API_KEY'

!pip uninstall -y pinecone
!pip -q install openai \
  pinecone==6.0.1 \
  pandas==2.2.2 \
  pinecone[grpc] \
  pinecone[asyncio]
# from crewai_tools import Tool
from langchain.tools import BaseTool
import json
import requests
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Pinecone


from pinecone import (
    Pinecone,
    ServerlessSpec,
    CloudProvider,
    AwsRegion,
    VectorType
)

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'
COMPOSIO_API_KEY = userdata.get('COMPOSIO_API_KEY') or 'COMPOSIO_API_KEY'
toolset = ComposioToolSet(api_key=COMPOSIO_API_KEY)

tools = toolset.get_tools(apps=[App.NOTION])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.1 MB/s eta 0:00:00


INFO:composio.utils.shared:Actions cache is outdated, refreshing cache...


In [27]:
# Create a serverless index
index_name = "huey"

pc = Pinecone(api_key=PINECONE_API_KEY)

# Target the index where you'll store the vector embeddings
index = pc.Index(index_name)

records = []
namespace = ""

client = OpenAI(
    # This is the default and can be omitted
    api_key=userdata.get('OPENAI_API_KEY') or 'OPENAI_API_KEY',
)

# CrewAI tools

In [33]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from crewai import Crew, Process, Task, Agent
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
import random
from typing import Any, Dict, Union, Type, List
import json
import requests
import os


# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()


class LinkedInSearchToolInput(BaseModel):
    """Input schema for MyCustomTool."""
    linkedin_url: str = Field(..., description="LinkedIn profile URL.")

# LinkedInSearchTool
class LinkedInSearchTool(BaseTool):
    name: str = "linkedin_search_tool"
    description: str = "search for LinkedIn profile and extract information"
    args_schema: Type[BaseModel] = LinkedInSearchToolInput

    def _run(self, linkedin_url: str):
        try:
            # Parse the raw profile data (whether it's HTML or JSON from another source)
            if isinstance(linkedin_url, str):
                try:
                  proxycurl_api_key = userdata.get('PROXYCURL_API_KEY') or 'PROXYCURL_API_KEY'
                  api_endpoint = 'https://nubela.co/proxycurl/api/v2/linkedin'
                  linkedin_profile_url = linkedin_url
                  headers = {'Authorization': 'Bearer ' + proxycurl_api_key}

                  response = requests.get(api_endpoint,
                        params={'url': linkedin_profile_url,'skills': 'include'},
                        headers=headers)

                  profile_data = response.json()

                except json.JSONDecodeError:
                    # If it's not JSON, assume it's HTML or other raw format
                    # In a real implementation, you'd have more sophisticated parsing
                    return json.dumps({"error": "Received non-JSON data, parsing not implemented in MVP"})

            # Extract and structure the key components
            structured_profile = {
                "summary": profile_data['summary'],
                "basic_info": profile_data['full_name'] + profile_data['occupation'] + profile_data['headline'],
                "work_experience": profile_data['experiences'],
                "education": profile_data['education'],
                "skills": profile_data['skills'],
                "achievements": profile_data['accomplishment_courses'],
                "recommendations": profile_data['recommendations']
            }

            # Remove any empty sections
            structured_profile = {k: v for k, v in structured_profile.items() if v}

            return json.dumps(structured_profile)

        except Exception as e:
            return json.dumps({"error": f"Profile parsing failed: {str(e)}"})

class SaveToNotionToolInput(BaseModel):
    """Input schema for SaveToNotionTool."""
    name: str = Field(..., description="Full name of the user")
    email: str = Field(..., description="Email address of the user")
    linkedin_url: str = Field(..., description="LinkedIn profile URL of the user")
    report: str = Field(..., description="Extracted profile report content")
    tone_of_voice: str = Field(..., description="Preferred tone of voice for the AI assistant")
    id: str = Field(..., description="Unique identifier for the user")
    goal: str = Field(..., description="Primary goal or objective of the user")

class SaveToNotionTool(BaseTool):
    name: str = "save_to_notion_tool"
    description: str = "Save the extracted profile data to Notion database via API"
    args_schema: Type[BaseModel] = SaveToNotionToolInput

    def _run(self, name: str, email: str, linkedin_url: str, report: str, tone_of_voice: str, id: str, goal: str):
        try:
            # Prepare the request data
            request_data = {
                "name": name,
                "email": email,
                "linkedinUrl": linkedin_url,
                "report": report,
                "toneOfVoice": tone_of_voice,
                "user_id": id,
                "goal": goal
            }

            # API endpoint URL
            api_url = "https://ti6bslxjq1.execute-api.us-east-1.amazonaws.com/Prod/"

            # Make the POST request
            response = requests.post(
                url=api_url,
                json=request_data,
                headers={"Content-Type": "application/json"}
            )

            # Check if the response is 500
            if response.status_code == 500:
                response_data = response.json()

                # Get the page id so that we can add the report to the body of the page
                notion_page_id = response_data.get("notionPageId")

                # Now append the full report using Notion API
                if notion_page_id:
                    append_result = self._append_full_report_to_notion(notion_page_id, report)

                    return json.dumps({
                        "status": "success",
                        "message": "Data successfully saved to Notion with full report",
                        "response": response_data,
                        "append_result": append_result
                    })
                else:
                    return json.dumps({
                        "status": "warning",
                        "message": "Data saved but no page ID returned",
                        "response": response_data
                    })
            else:
                return json.dumps({
                    "status": "error",
                    "message": f"Failed to save data to Notion: HTTP {response.status_code}",
                    "response": response.json() if response.content else {}
                })

        except Exception as e:
            return json.dumps({
                "status": "error",
                "message": f"Failed to save data to Notion: {str(e)}"
            })

    def _append_full_report_to_notion(self, page_id: str, report: str) -> Dict:
        """Append the full report to Notion page using the API"""
        try:
            # Get the Notion API token (you'll need to set this as an environment variable)
            notion_token = userdata.get('NOTION_API_TOKEN') or os.environ.get('NOTION_API_TOKEN')

            if not notion_token:
                return {"status": "error", "message": "Notion API token not found"}

            # Prepare headers
            headers = {
                'Authorization': f'Bearer {notion_token}',
                'Content-Type': 'application/json',
                'Notion-Version': '2022-06-28'
            }

            # API endpoint for appending blocks
            url = f'https://api.notion.com/v1/blocks/{page_id}/children'

            # Convert report to Notion blocks
            blocks = self._convert_report_to_blocks(report)

            # Make the API call to append blocks
            response = requests.patch(url, headers=headers, json={"children": blocks})

            if response.status_code == 200:
                return {"status": "success", "blocks_added": len(blocks)}
            else:
                return {
                    "status": "error",
                    "message": f"Failed to append blocks: {response.status_code}",
                    "error": response.json() if response.content else {}
                }

        except Exception as e:
            return {"status": "error", "message": f"Failed to append report: {str(e)}"}


    def _convert_report_to_blocks(self, report: str) -> List[Dict]:
        """Convert report text to Notion blocks"""
        blocks = []

        # Add a header for the full report
        blocks.append({
            "object": "block",
            "type": "heading_2",
            "heading_2": {
                "rich_text": [
                    {
                        "type": "text",
                        "text": {
                            "content": "Complete Professional Profile Report"
                        }
                    }
                ]
            }
        })

        # Split the report into paragraphs
        paragraphs = report.split('\n\n')

        for paragraph in paragraphs:
            if not paragraph.strip():
                continue

            # If the paragraph is a heading (starts with #), convert to heading block
            if paragraph.strip().startswith('### '):
                content = paragraph.replace('### ', '').strip()
                blocks.append({
                    "object": "block",
                    "type": "heading_3",
                    "heading_3": {
                        "rich_text": [
                            {
                                "type": "text",
                                "text": {
                                    "content": content
                                }
                            }
                        ]
                    }
                })
            elif paragraph.strip().startswith('## '):
                content = paragraph.replace('## ', '').strip()
                blocks.append({
                    "object": "block",
                    "type": "heading_2",
                    "heading_2": {
                        "rich_text": [
                            {
                                "type": "text",
                                "text": {
                                    "content": content
                                }
                            }
                        ]
                    }
                })
            elif paragraph.strip().startswith('# '):
                content = paragraph.replace('# ', '').strip()
                blocks.append({
                    "object": "block",
                    "type": "heading_1",
                    "heading_1": {
                        "rich_text": [
                            {
                                "type": "text",
                                "text": {
                                    "content": content
                                }
                            }
                        ]
                    }
                })
            else:
                # Handle potential bullet points or numbered lists
                lines = paragraph.split('\n')
                for line in lines:
                    if not line.strip():
                        continue

                    # Check for bullet points
                    if line.strip().startswith('- ') or line.strip().startswith('* '):
                        content = line.replace('- ', '').replace('* ', '').strip()
                        blocks.append({
                            "object": "block",
                            "type": "bulleted_list_item",
                            "bulleted_list_item": {
                                "rich_text": [
                                    {
                                        "type": "text",
                                        "text": {
                                            "content": content
                                        }
                                    }
                                ]
                            }
                        })
                    # Check for numbered lists
                    elif line.strip()[:1].isdigit() and line.strip()[1] == '.':
                        content = line.split('.', 1)[1].strip()
                        blocks.append({
                            "object": "block",
                            "type": "numbered_list_item",
                            "numbered_list_item": {
                                "rich_text": [
                                    {
                                        "type": "text",
                                        "text": {
                                            "content": content
                                        }
                                    }
                                ]
                            }
                        })
                    else:
                        # Regular paragraph
                        blocks.append({
                            "object": "block",
                            "type": "paragraph",
                            "paragraph": {
                                "rich_text": [
                                    {
                                        "type": "text",
                                        "text": {
                                            "content": line.strip()
                                        }
                                    }
                                ]
                            }
                        })

        return blocks

class VectorDBStorageToolInput(BaseModel):
    """Input schema for VectorEmbeddingTool."""
    input_data: Union[str, Dict] = Field(..., description="The compiled report")


class VectorDBStorageTool(BaseTool):
    name: str = "vector_db_storage_tool"
    description: str = "Store the compiled report in the vector database"
    args_schema: Type[BaseModel] = VectorDBStorageToolInput

    def _run(self, input_data):
        try:
            # Handle different input types
            if isinstance(input_data, dict):
                # If input is a dictionary, extract the 'description' field if it exists
                if 'description' in input_data:
                    data = input_data['description']
                else:
                    # Convert dict to string if no description field
                    data = json.dumps(input_data)
            elif isinstance(input_data, str):
                # Try to parse if it's a JSON string
                try:
                    parsed_data = json.loads(input_data)
                    if isinstance(parsed_data, dict):
                        if 'input_data' in parsed_data and isinstance(parsed_data['input_data'], dict):
                            if 'description' in parsed_data['input_data']:
                                data = parsed_data['input_data']['description']
                            else:
                                data = json.dumps(parsed_data['input_data'])
                        else:
                            data = input_data
                    else:
                        data = input_data
                except json.JSONDecodeError:
                    # Not valid JSON, use as is
                    data = input_data
            else:
                # For any other type, convert to string
                data = str(input_data)

            print(f"Processing data for vectorization. Data type: {type(data)}")

            # Generate embedding
            result = pc.inference.embed(
                model="multilingual-e5-large",
                inputs=[data],
                parameters={"input_type": "passage", "truncate": "END"}
            )
            embedding = result.data[0]['values']
            print(f"Embedding generated successfully. Vector length: {len(embedding)}")

            # Create a unique ID for this vector
            id_number = str(random.randint(10000000, 99999999))

            # Prepare the vector with metadata
            vector = {
                'id': id_number,
                'values': embedding,
                'metadata': {
                    'text': data
                }
            }

            # Use the ID as the namespace
            namespace = id_number

            # Upsert the vector to Pinecone
            index.upsert(vectors=[vector], namespace=namespace)
            print(f"Successfully stored vector with ID: {id_number} in namespace: {namespace}")

            return f"Successfully stored profile data with ID: {id_number} in namespace: {namespace}"

        except Exception as e:
            print(f"Error in vector storage: {str(e)}")
            return json.dumps({"error": f"Failed to store in vector database: {str(e)}"})

# Register tools
linkedin_search_tool = LinkedInSearchTool()
db_storage_tool = VectorDBStorageTool()
save_to_notion_tool = SaveToNotionTool()

# Agents

# Doer agents
These agents are agents that get things done with no delegation abaility. They are a bit like juniors who do a lot of the heavy low level work like transforming, searching of data etc.

In [29]:
# LinkedIn Data Scraper Agent
linkedin_scraper = Agent(
    role="LinkedIn Data Scraper",
    goal="Extract comprehensive profile information from LinkedIn URLs",
    backstory="I am an expert at retrieving and parsing professional data from LinkedIn profiles. I can extract work history, education, skills, and other relevant information.",
    verbose=True,
    allow_delegation=False,
    tools=[linkedin_search_tool]
)

# Report Compiler Agent
report_compiler = Agent(
    role="Professional Profile Reporter",
    goal="Create comprehensive, well-structured reports about professionals based on their career data  and save them to Notion",
    backstory="I transform raw professional data into coherent narratives that highlight expertise, qualifications, and career achievements, and ensure they're properly stored.",
    verbose=True,
    allow_delegation=False,
    tools=[save_to_notion_tool]
)

# Vectorization Agent
vectorization_agent = Agent(
    role="Data Vectorization Specialist",
    goal="Transform professional reports into optimized vector embeddings for retrieval",
    backstory="I prepare data for efficient storage and retrieval in vector databases, ensuring the AI assistant can access relevant information quickly.",
    verbose=True,
    allow_delegation=False,
    tools=[db_storage_tool]
)



# Heads / Lead Agents
These are agents that are specialisits and have the ability to delegate to doer agents to get the information they need to process and make decisions.

In [30]:
# Lead Company Research Agent
company_researcher = Agent(
    role="Lead Company Research Specialist",
    goal="Gather detailed information about companies mentioned in professional profiles",
    backstory="I specialize in researching organizations to understand their industry position, culture, and relevance to a professional's career trajectory.",
    verbose=True,
    allow_delegation=True,
    tools=[search_tool, scrape_tool]
)

# Lead Career Analyzer Agent
career_analyzer = Agent(
    role="Lead Career Pattern Analyst",
    goal="Identify patterns, progression, and expertise areas from career history",
    backstory="Organized and detail-oriented, you are an experienced career expert who analyzes professional trajectories to extract insights about specializations, growth patterns, and core competencies.",
    verbose=True,
    allow_delegation=False,
    tools=[search_tool]
)



# Tasks

In [31]:
# Task 1: Extract LinkedIn Profile Data
extract_profile_data = Task(
    description="""
    Extract comprehensive information from the LinkedIn profile URL:
    1. Full name and current position
    2. Complete work history with dates, companies, and role descriptions and responsibilities in each
    3. Education history with institutions, degrees, and fields of study
    4. Skills and endorsements
    5. Publications, patents, or other professional accomplishments
    6. Volunteering experience and causes
    7. Recommendations and testimonials if available

    The LinkedIn URL is: {linkedin_url}
    """,
    agent=linkedin_scraper,
    expected_output="Structured JSON containing all available profile information"
)

# Task 2: Research Companies
research_companies = Task(
    description="""
    Research all companies mentioned in the professional's work history:
    1. Company overview, industry, and size
    2. Main products or services
    3. Notable achievements or market position
    4. Company culture and values if available

    """,
    agent=company_researcher,
    expected_output="Dictionary of company profiles with key insights",
    context=[extract_profile_data]
)

# Task 3: Analyze Career Patterns
analyze_career = Task(
    description="""
    Analyze the professional's career trajectory to identify:
    1. Core expertise areas and specializations
    2. Career progression patterns
    3. Industry focus and possible pivots
    4. Leadership experience
    5. Notable achievements and their significance

    Use the profile data and company research to inform your analysis.
    """,
    agent=career_analyzer,
    expected_output="Analysis report of career patterns, expertise, and notable aspects",
    context=[extract_profile_data, research_companies]
)

# Task 4: Compile Comprehensive Report
compile_report = Task(
    description="""
    Create a comprehensive report about the professional that:
    1. Summarizes their current role and expertise
    2. Details their career progression with key achievements
    3. Highlights educational background and qualifications
    4. Identifies key skills and competencies
    5. Notes any specializations or unique professional attributes
    6. Contextualizes their experience within their industry

    The report should be well-structured, factual, and focused on information that would be valuable
    for networking connections wanting to learn more about them.

    Format the report in markdown for readability.
    """,
    agent=report_compiler,
    expected_output="Comprehensive professional report in markdown format",
    context=[extract_profile_data, research_companies, analyze_career]
)

# Task 5: Vectorize Report for Database
vectorize_report = Task(
    description="""
    Take the compiled professional report text and store it in the vector database.
    This is important: When you store the report in the vector database, you MUST return the generated ID
    that was used as the namespace, as this ID will be needed in the next task.

    The report data is provided in the compiled report from the previous task.

    Make sure to properly extract and process the report for vector storage, and return the unique ID
    that was generated for tracking purposes.
    """,
    agent=vectorization_agent,
    expected_output="The unique ID used for storing the vector, along with confirmation of successful vectorization",
    context=[compile_report]
)

# Task 6: Save Report to Notion
save_report_to_notion = Task(
    description="""
    Save the compiled report to Notion with all necessary metadata:
    1. Use the save_to_notion_tool to store the report
    2. Include the user's name, email, LinkedIn URL, generated report, tone of voice, user goal, and the unique ID generated in Task 5 (vectorize_report task)
    3. Set the appropriate tone of voice for future AI interactions
    4. You MUST use the unique vector ID from the previous task as the Id parameter

    Use this information:
    - Name: {name}
    - Email: {email}
    - LinkedIn URL: {linkedin_url}
    - Report: The full complete compiled report from Task 4
    - Id: The unique ID generated in Task 5 (vectorize_report task)
    - Tone of Voice: {tone_of_voice}
    - User Goal: {goal}

    IMPORTANT: Make sure to use the exact compiled report content as the "report" parameter, and the exact
    vector ID as the "id" parameter when calling the save_to_notion_tool.
    """,
    agent=report_compiler,
    expected_output="Confirmation that the report was successfully saved to Notion",
    context=[compile_report, vectorize_report]  # Include both the report and vector tasks as context
)

# Crew Definitions

In [34]:
linkedin_processing_crew = Crew(
    agents=[linkedin_scraper, company_researcher, career_analyzer, report_compiler, vectorization_agent],
    tasks=[extract_profile_data, research_companies, analyze_career, compile_report, vectorize_report, save_report_to_notion],
    verbose=True,
    process=Process.sequential  # Process tasks in order
)

# Function to initiate the crew with form data
def process_linkedin_profile():
    result = linkedin_processing_crew.kickoff(
        inputs={
            "linkedin_url": "https://www.linkedin.com/in/ethanrichardson/",
            "email": "ethan.b.richardson@gmail.com",
            "name": "Ethan Richardson",
            "tone_of_voice": "Professional, straight talking and clear. I want an approachable tone.",
            "goal": "I want to promote huey which is my new start-up that aims to help make following up after networking events easier. I want to be seen as professional and also a thought leader in the AI agent space."
        }
    )
    return result

process_linkedin_profile()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fb1f28de-be50-4b50-b0e3-f132f77f700b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 49267f65-8bcd-460a-9948-194df4b0f410
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 49267f65-8bcd-460a-9948-194df4b0f410
       Status: Executing Task...
    └── 🤖 Agent: LinkedIn Data Scraper
            Status: In Progress

# Agent: LinkedIn Data Scraper
## Task: 
    Extract comprehensive information from the LinkedIn profile URL:
    1. Full name and current position
    2. Complete work history with dates, companies, and role descriptions and responsibilities in each
    3. Education history with institutions, degrees, and fields of study
    4. Skills and endorsements
    5. Publications, patents, or other professional accomplishments
    6. Volunteering experience and causes
    7. Recommendations and testimonials if available
    
    The LinkedIn URL is: https://www.linkedin.com/in/ethanrichardson/
    


🤖 Agent: LinkedIn Data Scraper
    Status: In Progress

...

🚀 Crew: crew
├── 📋 Task: 49267f65-8bcd-460a-9948-194df4b0f410
│      Assigned to: LinkedIn Data Scraper
│      Status: ✅ Completed
│   └── 🤖 Agent: LinkedIn Data Scraper
│           Status: ✅ Completed
├── 📋 Task: bb14269f-01db-43a7-bd94-37aaf410f8a6
│      Assigned to: Lead Company Research Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Company Research Specialist
│           Status: ✅ Completed
├── 📋 Task: e1549301-4cd5-4967-8e74-830a5df82d3a
│      Assigned to: Lead Career Pattern Analyst
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Career Pattern Analyst
│           Status: ✅ Completed
├── 📋 Task: 1adc45b8-de29-44d8-a02a-985acd686a25
│      Assigned to: Professional Profile Reporter
│      Status: ✅ Completed
│   └── 🤖 Agent: Professional Profile Reporter
│           Status: ✅ Completed
├── 📋 Task: fb2f4e7c-eee5-4582-ba9b-7c06771f211b
│      Assigned to: Data Vectorization Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Data Vectorization Specialist
│           Status: ✅ Completed
└── 📋 Task: 83325483-6f83-40fc-a25d-237940d9d088
       Status: Executing Task...
    └── 🤖 Agent: Professional Profile Reporter
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 49267f65-8bcd-460a-9948-194df4b0f410
│      Assigned to: LinkedIn Data Scraper
│      Status: ✅ Completed
│   └── 🤖 Agent: LinkedIn Data Scraper
│           Status: ✅ Completed
├── 📋 Task: bb14269f-01db-43a7-bd94-37aaf410f8a6
│      Assigned to: Lead Company Research Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Company Research Specialist
│           Status: ✅ Completed
├── 📋 Task: e1549301-4cd5-4967-8e74-830a5df82d3a
│      Assigned to: Lead Career Pattern Analyst
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Career Pattern Analyst
│           Status: ✅ Completed
├── 📋 Task: 1adc45b8-de29-44d8-a02a-985acd686a25
│      Assigned to: Professional Profile Reporter
│      Status: ✅ Completed
│   └── 🤖 Agent: Professional Profile Reporter
│           Status: ✅ Completed
├── 📋 Task: fb2f4e7c-eee5-4582-ba9b-7c06771f211b
│      Assigned to: Data Vectorization Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Data Vectorization Specialist
│           Status: ✅ Completed
└── 📋 Task: 83325483-6f83-40fc-a25d-237940d9d088
       Assigned to: Professional Profile Reporter
       Status: ✅ Completed
    └── 🤖 Agent: Professional Profile Reporter
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 83325483-6f83-40fc-a25d-237940d9d088                                                                     │
│  Agent: Professional Profile Reporter                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fb1f28de-be50-4b50-b0e3-f132f77f700b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw="Ethan Richardson's professional profile has been successfully saved to Notion, including his name, email, LinkedIn URL, a concise compiled report of his career, preferred tone of voice, user goal, and the unique ID. The report was truncated to fit within the character limits required by Notion while retaining its essential elements.", pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='\n    Extract comprehensive information from the LinkedIn profile URL:\n    1. Full name and current position\n    2. Complete work history with dates, companies, and role descriptions and responsibilities in each\n    3. Education history with institutions, degrees, and fields of study\n    4. Skills and endorsements\n    5. Publications, patents, or other professional accomplishments\n    6. Volunteering experience and causes\n    7. Recommendations and testimonials if available\n    \n    The LinkedIn URL is: https://www.linkedin.com/in/ethanrichardson/\n    ', name=No